# 02 — CTC, Attention Alignment, and Whisper

## Why Not Always Use Forced Alignment?

Forced alignment (DTW/MFA) requires knowing the **transcript in advance**.
For real ASR, the whole point is to discover the transcript from audio.

Two solutions developed over time:

| | DTW / MFA | CTC | Attention (Whisper) |
|--|-----------|-----|---------------------|
| Needs transcript at inference | Yes | No | No |
| Learns alignment | No (hardcoded DP) | Implicitly | Explicitly via cross-attention |
| Training labels needed | Phoneme boundaries | (audio, text) only | (audio, text) only |
| Used in | FastSpeech2 data prep | DeepSpeech, wav2vec 2.0 | Whisper |
| Word timestamps | Yes (exact) | Approx | Yes (from attention weights) |

**Papers covered:**
- CTC — Graves et al., ICML 2006
- DeepSpeech 2 — Baidu, arXiv 1512.02595
- wav2vec 2.0 — Facebook AI, arXiv 2006.11477
- Whisper — OpenAI, arXiv 2212.04356

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn

np.random.seed(7)
torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. CTC — Connectionist Temporal Classification

**Paper:** Graves et al., ICML 2006

### The blank token trick

CTC adds a special `<blank>` token. The model outputs a sequence **longer** than the target text, then collapses it:

```
Target:          H   E   L   L   O
                 |   |   |   |   |
CTC raw output:  H H - E - L L - L - - O -
                         ^ blank

Step 1 — merge consecutive repeats:  H - E - L - L - O -
Step 2 — remove blanks:              H E L L O    ✓
```

The model never needs to know WHERE each character appears — CTC loss sums over **all valid paths** that collapse to the correct text.

### The blank enables two things
1. **Repeated characters**: "LL" in "hello" — without blank, LL would collapse to L
2. **Silence / held sounds**: frames where nothing new is being said

In [ ]:
def ctc_greedy_decode(log_probs, blank_id=0):
    pred = np.argmax(log_probs, axis=-1)
    merged = [pred[0]]
    for t in range(1, len(pred)):
        if pred[t] != merged[-1]:
            merged.append(pred[t])
    decoded = [tok for tok in merged if tok != blank_id]
    return pred, merged, decoded

vocab      = ['<blank>'] + list('abcdefghijklmnopqrstuvwxyz ')
vocab_size = len(vocab)
char_to_id = {c: i for i, c in enumerate(vocab)}

# Build a CTC-like output: model is confident about each char, blanks between
T = 30
log_probs = np.full((T, vocab_size), -10.0)
regions = [
    (0, 4,  'h'), (4, 7,  '<blank>'),
    (7, 12, 'e'), (12, 14, '<blank>'),
    (14, 18,'l'), (18, 20, '<blank>'),
    (20, 24,'l'), (24, 26, '<blank>'),
    (26, 30,'o'),
]
for start, end, char in regions:
    cid = char_to_id[char]
    log_probs[start:end, cid] = np.random.uniform(-0.5, 0.0, end - start)

raw_pred, merged, decoded = ctc_greedy_decode(log_probs, blank_id=0)
print("CTC Decode Steps:")
print(f"  Raw argmax:   {' '.join(vocab[i] for i in raw_pred)}")
print(f"  After merge:  {' '.join(vocab[i] for i in merged)}")
print(f"  After blank:  {''.join(vocab[i] for i in decoded)}")
print(f"  Target:       hello")
print(f"  Correct:      {'YES' if ''.join(vocab[i] for i in decoded) == 'hello' else 'NO'}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Top: log-prob heatmap (key tokens only)
show_ids    = [0] + [char_to_id[c] for c in 'hello']
show_labels = [vocab[i] for i in show_ids]
im = axes[0].imshow(log_probs[:, show_ids].T, aspect='auto', cmap='YlOrRd',
                     origin='lower', vmin=-5, vmax=0)
axes[0].set_yticks(range(len(show_ids)))
axes[0].set_yticklabels(show_labels, fontsize=11)
axes[0].set_xlabel('Audio frame')
axes[0].set_title('CTC Model Output — log P(token | frame)  [relevant tokens only]')
fig.colorbar(im, ax=axes[0])

# Bottom: frame-by-frame argmax
color_map = {'<blank>': '#CCCCCC', 'h': '#FF6B6B', 'e': '#4ECDC4',
             'l': '#45B7D1', 'o': '#96CEB4'}
for t in range(T):
    ch = vocab[raw_pred[t]]
    axes[1].add_patch(plt.Rectangle((t, 0), 1, 0.8,
                                     color=color_map.get(ch, '#FFE66D'), ec='gray', lw=0.5))
    axes[1].text(t + 0.5, 0.4, ch if ch != '<blank>' else '-',
                 ha='center', va='center', fontsize=9)
axes[1].set_xlim(0, T); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Audio frame')
axes[1].set_title('CTC Raw Frame Predictions  (grey = blank)')
patches = [mpatches.Patch(color=col, label=ch) for ch, col in color_map.items()]
axes[1].legend(handles=patches, loc='upper right', ncol=5, fontsize=9)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. CTC Loss in PyTorch

`nn.CTCLoss` computes the negative log-likelihood **summed over all valid alignment paths**.

This is why the model never needs boundary labels — during training it automatically discovers which alignment makes the target text most likely.

In [ ]:
B, T_in, vocab_size = 4, 50, 28

# IMPORTANT: CTC expects (T, B, C) — time dimension FIRST
log_probs      = torch.randn(T_in, B, vocab_size).log_softmax(dim=-1)
targets        = torch.randint(1, vocab_size, (B, 6))
input_lengths  = torch.full((B,), T_in, dtype=torch.long)
target_lengths = torch.randint(3, 7, (B,), dtype=torch.long)

ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
print(f"CTC Loss: {loss.item():.4f}")
print()
print("Shape convention:")
print(f"  log_probs:      {tuple(log_probs.shape)}   <- (T, B, vocab)  TIME first!")
print(f"  targets:        {tuple(targets.shape)}       <- (B, S)")
print(f"  input_lengths:  {tuple(input_lengths.shape)}         <- frames per sample")
print(f"  target_lengths: {tuple(target_lengths.shape)}         <- tokens per sample")
print()
print(f"Constraint: input_length >= target_length (need blanks between chars)")
print(f"  {T_in} frames >= max target {target_lengths.max().item()} tokens — OK")

## 3. From CTC to wav2vec 2.0

CTC is the foundation, but modern systems add a **self-supervised pre-training** stage.

**wav2vec 2.0** (Facebook AI, 2020) pipeline:

```
Stage 1 — Self-supervised pre-training (no labels needed):
  Raw audio -> CNN encoder -> context Transformer
  Objective: predict quantized audio codes from masked frames (like BERT for audio)
  Trained on 960h LibriSpeech unlabeled audio

Stage 2 — Fine-tuning with CTC (small labeled set):
  Pre-trained features -> linear layer -> CTC loss
  Only 10 minutes of labeled audio needed for competitive WER!
```

This shows why CTC is powerful: with good pre-trained features, very little labeled data is needed.

| wav2vec 2.0 | Labeled data | WER (test-clean) |
|-------------|-------------|-----------------|
| Large (no fine-tune) | 0 min | — |
| Large | 10 min | 4.8% |
| Large | 1 hour | 2.7% |
| Large | 960 hours | **1.8%** |

## 4. Attention-Based Alignment — Whisper Architecture

**Paper:** Radford et al., OpenAI 2022 (arXiv 2212.04356)

Whisper is a **Transformer encoder-decoder** trained on 680,000 hours of weakly supervised audio.

<img src="figures/whisper-arch.png" title="Whisper Architecture" style="width: 680px;" />

```
Audio (Log-Mel 80-bin, 30-sec chunks = 3000 frames)
         |
   CNN feature extractor  (2x conv, stride 2 -> 1500 frames)
         |
   Transformer Encoder  (audio representations)
         |
         +<────────── cross-attention ──────────+
         |                                       |
   Transformer Decoder                    text tokens so far
         |
   next token (BPE, same vocab as GPT-2)
```

### Key design choices
- **Multitask**: trained for transcription, translation, language ID, VAD — single model
- **Language**: 99 languages, zero-shot multilingual
- **Robustness**: trained on internet audio (noisy, accented, far-field) — very robust
- **No CTC**: purely cross-attention based alignment

In [ ]:
# Simulate Whisper-style cross-attention alignment
tokens       = ["<|startoftranscript|>", "hello", "world", "how", "are", "you", "<|endoftext|>"]
n_tokens     = len(tokens)
n_frames     = 80
true_centers = [0, 10, 25, 38, 50, 62, 75]

attn = np.zeros((n_tokens, n_frames))
for i, center in enumerate(true_centers):
    for f in range(n_frames):
        attn[i, f] = np.exp(-0.5 * ((f - center) / 5) ** 2)
    attn[i] += 0.015 * np.random.rand(n_frames)
    attn[i] /= attn[i].sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Top: attention heatmap
im = axes[0].imshow(attn, aspect='auto', cmap='Blues', origin='upper')
axes[0].set_yticks(range(n_tokens))
axes[0].set_yticklabels(tokens, fontsize=10)
axes[0].set_xlabel('Audio encoder frame')
axes[0].set_title('Whisper Cross-Attention Weights — which audio frame does each token attend to?')
fig.colorbar(im, ax=axes[0])
for i in range(n_tokens):
    axes[0].plot(np.argmax(attn[i]), i, 'r*', markersize=12)

# Bottom: word timeline extracted from attention peaks
colors = plt.cm.tab10(np.linspace(0, 1, n_tokens))
for i, (tok, center) in enumerate(zip(tokens, true_centers)):
    if tok in ('<|startoftranscript|>', '<|endoftext|>'): continue
    t_s = (center - 6) / n_frames * 30
    t_e = (center + 6) / n_frames * 30
    axes[1].add_patch(plt.Rectangle((t_s, 0.15), t_e - t_s, 0.7,
                                     color=colors[i], alpha=0.85, ec='black', lw=1.2))
    axes[1].text((t_s + t_e)/2, 0.5, tok, ha='center', va='center',
                 fontsize=11, fontweight='bold')
axes[1].set_xlim(0, 30); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Word Timestamps extracted from cross-attention peak frames')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print("Extracted timestamps:")
print(f"  {'Token':<10} {'Start':>8} {'End':>8}")
print("  " + "-" * 30)
for tok, center in zip(tokens, true_centers):
    if tok in ('<|startoftranscript|>', '<|endoftext|>'): continue
    print(f"  {tok:<10} {(center-6)/n_frames*30:>7.2f}s {(center+6)/n_frames*30:>7.2f}s")

## 5. Whisper in Practice

Whisper models come in five sizes:

| Model | Parameters | VRAM | Speed (relative) | WER (test-clean) |
|-------|-----------|------|-----------------|-----------------|
| tiny | 39M | ~1 GB | 32x | ~5.7% |
| base | 74M | ~1 GB | 16x | ~4.2% |
| small | 244M | ~2 GB | 6x | ~3.0% |
| medium | 769M | ~5 GB | 2x | ~2.8% |
| large-v3 | 1550M | ~10 GB | 1x | **~2.7%** |

All models support **99 languages** and multiple tasks (transcribe / translate).

In [ ]:
# ── Whisper practical usage ──
# pip install openai-whisper torchaudio jiwer

# import whisper

# 1. Load model
# model = whisper.load_model("base")   # tiny / base / small / medium / large

# 2. Transcribe an audio file (any format: mp3, wav, m4a, ...)
# result = model.transcribe("audio.mp3")
# print(result["text"])

# 3. With language hint (faster, skips language detection)
# result = model.transcribe("audio.mp3", language="en")

# 4. With word timestamps (uses attention weights internally)
# result = model.transcribe("audio.mp3", word_timestamps=True)
# for segment in result["segments"]:
#     for word in segment["words"]:
#         print(f"{word['word']:15s}  {word['start']:.2f}s - {word['end']:.2f}s")

# 5. Translation to English (from any language)
# result = model.transcribe("thai_audio.mp3", task="translate")

# 6. Multilingual — force Thai transcription
# result = model.transcribe("thai_audio.mp3", language="th")

print("Whisper API overview (requires: pip install openai-whisper)")
print()
print("Key parameters for model.transcribe():")
params = [
    ("language",         "str / None",  "Force language ('en','th','zh',...) or auto-detect"),
    ("task",             "str",         "'transcribe' (default) or 'translate' (to English)"),
    ("word_timestamps",  "bool",        "Return per-word timestamps from cross-attention"),
    ("beam_size",        "int",         "Beam search width (default 5); higher = more accurate"),
    ("temperature",      "float",       "Sampling temperature; 0.0 = greedy"),
    ("initial_prompt",   "str",         "Hint text to guide transcription style/vocabulary"),
    ("fp16",             "bool",        "Use float16 (faster on GPU, default True if CUDA)"),
]
print(f"  {'Parameter':<18} {'Type':<12} Description")
print("  " + "-" * 65)
for name, typ, desc in params:
    print(f"  {name:<18} {typ:<12} {desc}")

## 6. Evaluating Whisper with WER

A real evaluation pipeline:
1. Load a benchmark dataset (e.g. LibriSpeech test-clean)
2. Run Whisper inference on each audio clip
3. Normalize text (remove punctuation, lowercase)
4. Compute WER between hypothesis and reference

```python
# Full evaluation pipeline (requires whisper + jiwer + torchaudio)
import whisper, jiwer, torchaudio
from whisper.normalizers import EnglishTextNormalizer

model      = whisper.load_model("base")
normalizer = EnglishTextNormalizer()

# dataset: list of (audio_path, reference_text) pairs
hypotheses, references = [], []
for audio_path, reference in dataset:
    result = model.transcribe(audio_path, language="en")
    hypotheses.append(normalizer(result["text"]))
    references.append(normalizer(reference))

wer = jiwer.wer(references, hypotheses)
print(f"WER: {wer * 100:.2f}%")
```

The `EnglishTextNormalizer` handles:
- Lowercasing
- Removing punctuation
- Expanding contractions ("don't" -> "do not")
- Normalizing numbers ("$50" -> "fifty dollars")

In [ ]:
# Simulate WER evaluation results across Whisper model sizes
model_sizes  = ["tiny", "base", "small", "medium", "large-v3"]
wer_clean    = [5.7, 4.2, 3.0, 2.8, 2.7]    # LibriSpeech test-clean
wer_other    = [15.1, 11.3, 8.2, 6.9, 5.2]  # LibriSpeech test-other (noisier)
params_M     = [39, 74, 244, 769, 1550]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: WER vs model size
x = np.arange(len(model_sizes))
w = 0.35
axes[0].bar(x - w/2, wer_clean, w, label='test-clean', color='#5DADE2')
axes[0].bar(x + w/2, wer_other, w, label='test-other', color='#E59866')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_sizes)
axes[0].set_ylabel('WER (%)')
axes[0].set_title('Whisper WER by Model Size
(LibriSpeech)')
axes[0].legend()
axes[0].set_ylim(0, 18)
for i, (c, o) in enumerate(zip(wer_clean, wer_other)):
    axes[0].text(i - w/2, c + 0.3, f'{c}%', ha='center', fontsize=8)
    axes[0].text(i + w/2, o + 0.3, f'{o}%', ha='center', fontsize=8)

# Right: WER vs parameters (log scale)
axes[1].semilogx(params_M, wer_clean, 'o-', color='#5DADE2', lw=2,
                  markersize=9, label='test-clean')
axes[1].semilogx(params_M, wer_other, 's-', color='#E59866', lw=2,
                  markersize=9, label='test-other')
for i, (p, c, o) in enumerate(zip(params_M, wer_clean, wer_other)):
    axes[1].annotate(model_sizes[i], (p, c), textcoords="offset points",
                     xytext=(4, 6), fontsize=8)
axes[1].set_xlabel('Parameters (millions, log scale)')
axes[1].set_ylabel('WER (%)')
axes[1].set_title('WER vs Model Size (log scale)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("figures/whisper_wer.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved figures/whisper_wer.png")

## 7. CTC vs Attention — Side-by-Side

| | CTC | Attention (Whisper) |
|--|-----|---------------------|
| **Alignment type** | Hard per-frame label | Soft attention weights |
| **Decoding** | Greedy / beam over frames | Autoregressive token-by-token |
| **Blank token** | Yes — required | No |
| **Timestamps** | Approx (frame argmax) | Via attention peak (DTW on attention) |
| **Multilingual** | Needs separate models | Single model, 99 languages |
| **Training data** | Needs labeled audio | Weakly supervised (internet scale) |
| **Speed** | Faster (non-autoregressive) | Slower (autoregressive) |
| **Models** | DeepSpeech2, wav2vec 2.0 | Whisper |

In [ ]:
# Side-by-side alignment visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# CTC: hard discrete prediction per frame
ctc_seq = [0,0,1,1,0,2,2,2,0,3,3,0,4,4,4,0]  # 0=blank,1=H,2=E,3=L,4=O
labels  = {0:'-', 1:'H', 2:'E', 3:'L', 4:'O'}
colors  = {0:'#CCCCCC', 1:'#FF6B6B', 2:'#4ECDC4', 3:'#45B7D1', 4:'#96CEB4'}
for t, tok in enumerate(ctc_seq):
    axes[0].add_patch(plt.Rectangle((t, 0), 1, 0.8, color=colors[tok], ec='gray', lw=0.5))
    axes[0].text(t+0.5, 0.4, labels[tok], ha='center', va='center', fontsize=13)
axes[0].set_xlim(0, len(ctc_seq)); axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Audio frame'); axes[0].axis('off')
axes[0].set_title('CTC: hard per-frame label
(grey = blank token)')

# Attention: soft distribution over frames
n_f = 16
tok_names = ['H', 'E', 'L', 'O']
attn_soft = np.zeros((4, n_f))
for i, c in enumerate([2.5, 6, 9.5, 13]):
    for f in range(n_f):
        attn_soft[i, f] = np.exp(-0.5 * ((f - c) / 1.8)**2)
    attn_soft[i] /= attn_soft[i].sum()

im = axes[1].imshow(attn_soft, aspect='auto', cmap='Blues', origin='upper')
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(tok_names, fontsize=13)
axes[1].set_xlabel('Audio frame')
axes[1].set_title('Attention: soft weight over frames
(star = argmax / timestamp)')
fig.colorbar(im, ax=axes[1])
for i in range(4):
    axes[1].plot(np.argmax(attn_soft[i]), i, 'r*', markersize=14)

plt.suptitle('CTC (hard) vs Attention (soft) Alignment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("figures/ctc_vs_attention.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved figures/ctc_vs_attention.png")

## Summary

| | DTW / MFA | CTC | Attention (Whisper) |
|--|-----------|-----|---------------------|
| **Paper** | Classical | Graves 2006 | Radford 2022 |
| **Training labels** | Phoneme boundaries | (audio, text) | (audio, text) |
| **Transcript at inference** | Required | Not needed | Not needed |
| **Models** | MFA, forced alignment | DeepSpeech2, wav2vec 2.0 | Whisper |
| **Training scale** | Small | Medium | 680,000 hours |

**References:**
- CTC: [Graves et al. 2006](https://www.cs.toronto.edu/~graves/icml_2006.pdf)
- DeepSpeech 2: [arXiv 1512.02595](https://arxiv.org/abs/1512.02595)
- wav2vec 2.0: [arXiv 2006.11477](https://arxiv.org/abs/2006.11477)
- Whisper: [arXiv 2212.04356](https://arxiv.org/abs/2212.04356)